# RuSQL NL→SQL fine-tuning (Qwen2.5-Coder-1.5B-Instruct, LoRA/QLoRA)

Colab Pro notebook — needs a GPU runtime (Runtime -> Change runtime type -> T4/L4/A100).

**Before running:** upload `train.jsonl`, `val.jsonl`, `test.jsonl` from
`code/AI/data_generation/dataset/` in this repo into your Google Drive, under
`My Drive/projects/RuSQL/dataset/` (the config cell's `DATA_DIR` points there by default -
adjust it if you put them somewhere else). Uploading to Drive instead of Colab's ephemeral
`/content/` means they survive a runtime reset, so you don't have to re-upload every session.

Training saves the LoRA adapter to Google Drive (see `OUTPUT_DIR` in the config cell) so it
survives the Colab session ending — a plain `/content/...` path is wiped when the runtime recycles.

On a T4 (16GB, what free/Pro Colab usually gives you), leave `USE_4BIT = True`.
On an A100/L4 (>=24GB), you can set it `False` for a small quality/speed improvement.

## 1. Install dependencies

Pinned exactly — trl's `SFTTrainer`/`SFTConfig` API shape has changed across
releases; if a newer version resolves and a later cell errors on an unexpected
argument, re-run this cell to force these versions instead of chasing the new API.

In [ ]:
!pip install -q "torch>=2.3" transformers==4.46.3 peft==0.13.2 trl==0.12.1 bitsandbytes==0.46.1 accelerate==1.0.1 datasets==3.0.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.9/310.9 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.7/472.7 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 59.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does no

## 2. (Optional) mount Google Drive

Run this if you want the trained adapter to persist after the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Config

Adjust these paths/flags, then run everything below top to bottom.

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

DATA_DIR = "/content/drive/MyDrive/projects/RuSQL/dataset"  # upload train/val/test.jsonl here once (survives runtime resets)
TRAIN_FILE = f"{DATA_DIR}/train.jsonl"
VAL_FILE = f"{DATA_DIR}/val.jsonl"
TEST_FILE = f"{DATA_DIR}/test.jsonl"     # held-out schemas, used only for evaluation at the end

OUTPUT_DIR = "/content/drive/MyDrive/projects/RuSQL/rusql-nl2sql-lora"  # Drive path so it survives session end

USE_4BIT = True   # False if you have >=24GB VRAM (A100/L4) for slightly better quality
EPOCHS = 3.0
BATCH_SIZE = 2   # lowered from 8 - T4 (16GB) OOMs at 8 with this model/seq length
GRAD_ACCUM = 8   # raised from 2 so the effective batch size (BATCH_SIZE * GRAD_ACCUM) stays 16
LEARNING_RATE = 2e-4
MAX_SEQ_LEN = 512   # lowered from 1024 - our examples (schema + question + SQL) are short; frees more memory

## 4. Shared prompt format

Same message-building function to reuse later for local inference (kept identical
on purpose, so the model never sees a different prompt shape than it trained on).

In [ ]:
SYSTEM_PROMPT = (
    "You are a SQL assistant for the RuSQL database engine. Given a database schema "
    "and a question in Korean or English, output ONLY the SQL query that answers it. "
    "Do not explain, do not add commentary, do not wrap the query in markdown."
)


def build_messages(schema: str, question: str, sql: str | None = None) -> list[dict]:
    """Pass sql=None for an inference-time prompt; pass it to build a training example."""
    user_content = f"Schema:\n{schema}\n\nQuestion: {question}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    if sql is not None:
        messages.append({"role": "assistant", "content": sql})
    return messages

## 5. Load base model + tokenizer, attach LoRA

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = None
if USE_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype="bfloat16",
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()  # required for gradient checkpointing + LoRA to backprop correctly
model.print_trainable_parameters()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 6. Load + format the dataset

In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files={"train": TRAIN_FILE, "validation": VAL_FILE})


def _format(row: dict) -> dict:
    messages = build_messages(row["schema"], row["question"], row["sql"])
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}


dataset = raw.map(_format, remove_columns=raw["train"].column_names)
dataset["train"][0]["text"]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2993 [00:00<?, ? examples/s]

Map:   0%|          | 0/157 [00:00<?, ? examples/s]

'<|im_start|>system\nYou are a SQL assistant for the RuSQL database engine. Given a database schema and a question in Korean or English, output ONLY the SQL query that answers it. Do not explain, do not add commentary, do not wrap the query in markdown.<|im_end|>\n<|im_start|>user\nSchema:\nCREATE TABLE teacher (\n  id INT PRIMARY KEY,\n  name VARCHAR(50),\n  subject VARCHAR(50)\n);\nCREATE TABLE student (\n  id INT PRIMARY KEY,\n  name VARCHAR(50),\n  grade INT,\n  enrollment_date DATE\n);\nCREATE TABLE course (\n  id INT PRIMARY KEY,\n  name VARCHAR(50),\n  teacher_id INT REFERENCES teacher(id),\n  credit INT\n);\nCREATE TABLE enrollment (\n  id INT PRIMARY KEY,\n  student_id INT REFERENCES student(id),\n  course_id INT REFERENCES course(id),\n  score INT\n);\n\nQuestion: 등록일이 있는 학생만 알려줘<|im_end|>\n<|im_start|>assistant\nSELECT * FROM student WHERE enrollment_date IS NOT NULL;<|im_end|>\n'

## 7. Train

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=LEARNING_RATE,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    bf16=True,
    report_to=[],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Done. LoRA adapter saved to {OUTPUT_DIR}")

Map:   0%|          | 0/2993 [00:00<?, ? examples/s]

Map:   0%|          | 0/157 [00:00<?, ? examples/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Epoch,Training Loss,Validation Loss
0,0.061600,0.056946
1,0.050000,0.051194
2,0.044300,0.048746


Done. LoRA adapter saved to /content/drive/MyDrive/projects/RuSQL/rusql-nl2sql-lora


## 8. Evaluate on held-out schemas

`test.jsonl`'s schemas (`real_estate`/`airline`/`insurance`) were never used in
training, so this measures generalization to unseen schemas, not memorization.

This only checks string-level SQL match (exact and whitespace/case-normalized) — it
does **not** execute the generated SQL against a real RuSQL engine (unreachable from
Colab), so a generated query that's a reasonable paraphrase of the reference (e.g.
different but equivalent column order) still counts as "wrong" here. Treat this as a
rough progress signal, not a final accuracy number.

In [ ]:
import json
import re
from collections import Counter


def normalize_sql(sql: str) -> str:
    sql = sql.strip().rstrip(";").lower()
    return re.sub(r"\s+", " ", sql)


model.eval()
test_rows = [json.loads(line) for line in open(TEST_FILE, encoding="utf-8")]

exact = 0
normalized = 0
failures = []
for row in test_rows:
    messages = build_messages(row["schema"], row["question"])
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    if generated == row["sql"].strip():
        exact += 1
    if normalize_sql(generated) == normalize_sql(row["sql"]):
        normalized += 1
    else:
        failures.append({
            "domain": row.get("domain"),
            "lang": row.get("lang"),
            "question": row["question"],
            "expected": row["sql"],
            "generated": generated,
        })

n = len(test_rows)
print(f"rows evaluated: {n}")
print(f"exact match:      {exact}/{n} ({100 * exact / n:.1f}%)")
print(f"normalized match: {normalized}/{n} ({100 * normalized / n:.1f}%)")

# --- failure breakdown ---
domain_totals = Counter(row.get("domain") for row in test_rows)
domain_fails = Counter(f["domain"] for f in failures)
lang_totals = Counter(row.get("lang") for row in test_rows)
lang_fails = Counter(f["lang"] for f in failures)

print()
print(f"--- {len(failures)} failures by domain ---")
for domain, total in sorted(domain_totals.items()):
    fails = domain_fails.get(domain, 0)
    print(f"  {domain}: {fails}/{total} failed ({100 * fails / total:.1f}%)")

print()
print("--- failures by language ---")
for lang, total in sorted(lang_totals.items()):
    fails = lang_fails.get(lang, 0)
    print(f"  {lang}: {fails}/{total} failed ({100 * fails / total:.1f}%)")

print()
print("--- failure details ---")
for f in failures:
    print(f"[{f['domain']}/{f['lang']}] Q: {f['question']}")
    print(f"  expected:  {f['expected']}")
    print(f"  generated: {f['generated']}")
    print()